In [1]:
!pip install transformers torch librosa soundfile pydub

In [2]:
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import io

In [ ]:
import torch
import torchaudio
import io
import random


In [ ]:
AUDIO_PATH = './speech.wav'

# Load model wav2vec2 menggunakan torchaudio
print("Loading model, please wait...")
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [ ]:
def Transcriber(audiopath):
    # Kriteria: Predict Text
    waveform, sample_rate = torchaudio.load(audiopath)
    if sample_rate != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

    with torch.inference_mode():
        emissions, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emissions: torch.Tensor):
            idx = torch.argmax(emissions, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return "".join([self.labels[i] for i in idx])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    txt = decoder(emissions[0])

    # Mengganti karakter '|' dengan whitespace (' ') sesuai kriteria
    txt = txt.replace("|", " ")
    return txt

In [ ]:
def ExportMenu(audio_data):
    # Kriteria: Export File
    while True:
        filename = input("Input new file name: ")

        # Validasi akhiran nama file
        if not filename.endswith('.wav'):
            print("Filename must end with .wav")
        else:
            break

    # Menyimpan file berdasarkan tipe datanya (pydub AudioSegment atau tuple librosa)
    if isinstance(audio_data, AudioSegment):
        audio_data.export(filename, format='wav')
    else:
        # data dari librosa: (numpy_array, sample_rate)
        sf.write(filename, audio_data[0], audio_data[1])

    print("File exported successfully!")

In [ ]:
def AudioAugmentationMenu():
    # Kriteria: Load Data
    try:
        # Load via pydub untuk manipulasi durasi/efek
        audio_pydub = AudioSegment.from_file(AUDIO_PATH) # from_wav(AUDIO_PATH)
        # Load via librosa untuk pitch dan noise
        audio_librosa, sr_librosa = librosa.load(AUDIO_PATH, sr=None)
    except Exception as e:
        print("Error loading audio. Please make sure the file exists:", e)
        return

    while True:
        # Kriteria: Menu (Sub-menu)
        print("\nAudio Augmentation Menu")
        print("1. Pitch shift (Increase or decrease pitch)")
        print("2. Add white noise")
        print("3. Repeat clip")
        print("4. Insert fade in")
        print("5. Back to main menu")

        cc = input("Input your choice: ")

        if cc == "1":
            while True:
                try:
                    pitch = float(input("Input pitch rate: "))
                    # Validasi input pitch rate antara -5 dan 5
                    if -5 <= pitch <= 5:
                        break
                    else:
                        print("Pitch rate must be between -5 and 5.")
                except ValueError:
                    print("Invalid input format.")

            # Kriteria: Audio Augmentation (Pitch shift)
            shifted = librosa.effects.pitch_shift(y=audio_librosa, sr=sr_librosa, n_steps=pitch)
            ExportMenu((shifted, sr_librosa))

        elif cc == "2":
            while True:
                try:
                    noise_level = float(input("Input noise level: "))
                    # Validasi noise level antara 0.1 dan 0.5
                    if 0.1 <= noise_level <= 0.5:
                        break
                    else:
                        print("Noise level must be between 0.1 and 0.5.")
                except ValueError:
                    print("Invalid input format.")

            # Kriteria: Audio Augmentation (Add white noise)
            noise = np.random.randn(len(audio_librosa))
            augmented_audio = audio_librosa + noise_level * noise
            ExportMenu((augmented_audio, sr_librosa))

        elif cc == "3":
            while True:
                try:
                    repeat_times = int(input("Input how many times to repeat: "))
                    # Validasi repeat times lebih dari 0
                    if repeat_times > 0:
                        break
                    else:
                        print("Repeat times must be greater than 0.")
                except ValueError:
                    print("Invalid input format.")

            # Kriteria: Audio Augmentation (Repeat clip)
            repeated_audio = audio_pydub * repeat_times
            ExportMenu(repeated_audio)

        elif cc == "4":
            while True:
                try:
                    # Input durasi dalam milidetik untuk pydub (misal 3000 = 3 detik)
                    fade_dur = int(input("Input duration effect (in ms): "))
                    # Validasi durasi efek lebih dari 0
                    if fade_dur > 0:
                        break
                    else:
                        print("Duration must be more than 0.")
                except ValueError:
                    print("Invalid input format.")

            # Kriteria: Audio Augmentation (Insert fade in)
            faded_audio = audio_pydub.fade_in(fade_dur)
            ExportMenu(faded_audio)

        elif cc == "5":
            break
        else:
            print("Invalid Input")

In [ ]:
def MainMenu():
    while True:
        # Kriteria: Menu (Utama)
        print("\n1. Audio Augmentation")
        print("2. Predict text")
        print("3. Exit")

        cc = input("Input your choice: ")

        if cc == "1":
            AudioAugmentationMenu()
        elif cc == "2":
            try:
                txt = Transcriber(AUDIO_PATH)
                # Output sesuai format yang diminta
                print(f"Predicted audio text:  {txt}")
            except Exception as e:
                print("Error predicting text:", e)
        elif cc == "3":
            break
        else:
            print("Invalid Input")

In [ ]:
if __name__ == "__main__":
    MainMenu()